In [8]:
import numpy as np
import pandas as pd

# 1. INITIALIZATION:
# 'dp' table stores the minimum cost to reach city 'i' having visited subset 'mask'
# 'parent' table stores the previous city to allow path reconstruction later
df = pd.read_csv('distance_matrix_small (1).csv')
dist_matrix = df.iloc[:, 1:].values
n_total = len(dist_matrix)
n = n_total - 1  
dp = np.full((1 << n, n), float('inf'))
parent = np.full((1 << n, n), -1)


In [9]:
# Base case: Calculate distance from Riyadh (index 0) to every other city
# 1 << i represents a bitmask where only the i-th city is visited
for i in range(n):
        dp[1 << i][i] = dist_matrix[0][i+1]

In [10]:
# 2. CORE DYNAMIC PROGRAMMING LOOP:
# Iterate through all possible combinations of visited cities (subsets/masks)
for mask in range(1, 1 << n):
        for i in range(n):
            if not (mask & (1 << i)): continue 
            
            for j in range(n):
                if mask & (1 << j): continue 
                
                new_mask = mask | (1 << j)
                cost = dp[mask][i] + dist_matrix[i+1][j+1]
                if cost < dp[new_mask][j]:
                    dp[new_mask][j] = cost
                    parent[new_mask][j] = i



In [11]:
# 3. FINAL STEP: RETURN TO STARTING CITY (RIYADH)
# full_mask (all 1s) means all cities have been visited
full_mask = (1 << n) - 1
min_cost = float('inf')
last_city = -1

for i in range(n):
    cost = dp[full_mask][i] + dist_matrix[i+1][0]
    
    if cost < min_cost:
        min_cost = cost
        last_city = i

In [12]:
import pandas as pd

df = pd.read_csv('distance_matrix_small (1).csv')

city_names = df.columns[1:].tolist()
# 4. PATH RECONSTRUCTION (BACKTRACKING):
# We work backwards from the last city to the first to find the optimal order
path = [0]
curr_mask = full_mask
curr_city = last_city
temp_path = []

while curr_city != -1:
    temp_path.append(curr_city + 1)
    prev_city = parent[curr_mask][curr_city]
    curr_mask ^= (1 << curr_city)
    curr_city = prev_city

path.extend(reversed(temp_path))
path.append(0)

final_route_names = [city_names[idx] for idx in path]

print("--- Final Result ---")
print(f"Optimal Distance: {min_cost:.2f}")
print(f"Route: {' -> '.join(final_route_names)}")

--- Final Result ---
Optimal Distance: 309.21
Route: Riyadh -> Hail -> Medina -> Jeddah -> Mecca -> Dammam -> Abha -> Tabuk -> Khobar -> Riyadh


In [13]:
import pandas as pd
import numpy as np

def solve_tsp_dp(dist_matrix, city_names):
    n_total = len(dist_matrix)
    n = n_total - 1 
    
    dp = np.full((1 << n, n), float('inf'))
    parent = np.full((1 << n, n), -1)

    for i in range(n):
        dp[1 << i][i] = dist_matrix[0][i+1]

    for mask in range(1, 1 << n):
        for i in range(n):
            if not (mask & (1 << i)): continue
            for j in range(n):
                if mask & (1 << j): continue
                new_mask = mask | (1 << j)
                cost = dp[mask][i] + dist_matrix[i+1][j+1]
                if cost < dp[new_mask][j]:
                    dp[new_mask][j] = cost
                    parent[new_mask][j] = i

    full_mask = (1 << n) - 1
    min_cost = float('inf')
    last_city = -1
    for i in range(n):
        cost = dp[full_mask][i] + dist_matrix[i+1][0]
        if cost < min_cost:
            min_cost = cost
            last_city = i
            
    path_idx = [0]
    curr_m, curr_c = full_mask, last_city
    tmp = []
    while curr_c != -1:
        tmp.append(curr_c + 1)
        prev_c = parent[curr_m][curr_c]
        curr_m ^= (1 << curr_c)
        curr_c = prev_c
    path_idx.extend(reversed(tmp))
    path_idx.append(0)
    
    route_names = [city_names[idx] for idx in path_idx]

    return min_cost, route_names


df = pd.read_csv('distance_matrix_small (1).csv')
dist_matrix = df.iloc[:, 1:].values
city_names = df.columns[1:].tolist()

final_cost, final_route = solve_tsp_dp(dist_matrix, city_names)

# Display the final results
print(f"Shortest Distance: {final_cost:.2f}")
print(f"Optimal Route: {' -> '.join(final_route)}")

Shortest Distance: 309.21
Optimal Route: Riyadh -> Hail -> Medina -> Jeddah -> Mecca -> Dammam -> Abha -> Tabuk -> Khobar -> Riyadh


In [15]:
import time
# Best Case: Subset of 4 cities
small_dist = dist_matrix[:4, :4]
small_cities = city_names[:4]

start = time.time()
cost, route = solve_tsp_dp(small_dist, small_cities)
end = time.time()

print("--- BEST CASE (4 Cities) ---")
print(f"Route: {' -> '.join(route)}")
print(f"Distance: {cost:.2f}")
print(f"Time: {end - start:.6f} seconds")

--- BEST CASE (4 Cities) ---
Route: Riyadh -> Mecca -> Dammam -> Jeddah -> Riyadh
Distance: 204.53
Time: 0.003324 seconds


In [16]:
# Average Case: Subset of 7 cities
avg_dist = dist_matrix[:7, :7]
avg_cities = city_names[:7]

start = time.time()
cost, route = solve_tsp_dp(avg_dist, avg_cities)
end = time.time()

print("--- AVERAGE CASE (7 Cities) ---")
print(f"Route: {' -> '.join(route)}")
print(f"Distance: {cost:.2f}")
print(f"Time: {end - start:.6f} seconds")

--- AVERAGE CASE (7 Cities) ---
Route: Riyadh -> Khobar -> Abha -> Dammam -> Mecca -> Jeddah -> Medina -> Riyadh
Distance: 274.11
Time: 0.003329 seconds


In [17]:
# Worst Case: All 9 cities
start = time.time()
cost, route = solve_tsp_dp(dist_matrix, city_names)
end = time.time()

print("--- WORST CASE (9 Cities) ---")
print(f"Route: {' -> '.join(route)}")
print(f"Distance: {cost:.2f}")
print(f"Time: {end - start:.6f} seconds")

--- WORST CASE (9 Cities) ---
Route: Riyadh -> Hail -> Medina -> Jeddah -> Mecca -> Dammam -> Abha -> Tabuk -> Khobar -> Riyadh
Distance: 309.21
Time: 0.030609 seconds
